In [ ]:
%load_ext autoreload
%autoreload 2

import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.colors as mcl

from files import data, features

from collections import Counter

In [ ]:
# Make sure this matches project root every time

ROOT = '..'

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(ROOT)

os.environ["DATA_DIR"] = "datasets"

In [ ]:
listings = data.load("listings")

off_market = data.load("off_market")
list_aware = data.load("list_aware")

In [ ]:
', '.join(listings.columns)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (13, 5))

axes[0].hist(listings['sold_price'], bins = 100)
axes[0].set(title = 'sold_price')

axes[1].hist(np.log(listings['sold_price']), bins = 100, color = 'orange')
axes[1].set(title = 'log(sold_price)')

plt.suptitle('Distribution of sold_price for all listings (in $M)')
plt.show()

In [ ]:
plt.hist(np.log(listings['list_price']) - np.log(listings['sold_price']), bins = 70, color = 'teal', alpha = 0.6)

plt.title('Distribution of log(list_price/sold_price) for all listings')

plt.show()

In [ ]:
plt.scatter('list_price', 'sold_price', data = listings, marker = '+', color = 'magenta', alpha = 0.25)

plt.title('final sold_price vs. original list_price (in $M)')

plt.xlabel('list_price')
plt.ylabel('sold_price')

plt.show()

In [ ]:
avm_est_available = listings[listings['external_avm_estimate_date'] < listings['date_sold']]

print(f'{len(avm_est_available)} records with external_avm_estimate available before date_sold')
print(f'({np.round(len(avm_est_available) / len(listings) * 100, 2)}% of complete dataset)')

In [ ]:
plt.scatter('external_avm_estimate', 'sold_price', data = avm_est_available, marker = '+', color = 'lightgreen', alpha = 0.25)

plt.title(f'sold_price vs. external_avm_estimate for {len(avm_est_available)} records (in $M)')

plt.xlabel('external_avm_estimate')
plt.ylabel('sold_price')

plt.show()

In [ ]:
listings[['house_type_name', 'sold_price']].groupby('house_type_name').describe().round(2)

In [ ]:
listings[['municipality', 'sold_price']].groupby('municipality').describe().round(2)

In [ ]:
np.random.seed(91)

muni_counts = listings['municipality'].value_counts()

muni_colors = [mcl.to_hex(color) for color in np.random.rand(len(muni_counts), 3)]

muni_counts.sort_index().plot(kind = 'bar', rot = 0, color = muni_colors)

plt.title('Listing counts by municipality')
plt.xlabel(None)

plt.show()

print(muni_counts)

In [ ]:
for i, muni in enumerate(muni_counts.index):
    plt.scatter('latitude', 'longitude', data = listings[listings['municipality'] == muni],
                marker = '^', color = muni_colors[i], alpha = 0.025)

plt.title(f'Approximate map of listings in municipalities')

plt.xlabel('latitude')
plt.ylabel('longitude')

plt.show()

In [ ]:
plt.scatter('latitude', 'longitude', data = listings, marker = '^', color = 'green', alpha = 0.025)

plt.scatter('latitude', 'longitude', data = listings[listings['sold_price'] > listings['sold_price'].median()],
            marker = '^', color = 'magenta', alpha = 0.0125)

plt.title(f'Approximate map of listings')

plt.xlabel('latitude')
plt.ylabel('longitude')

plt.show()

In [ ]:
month_group = listings['date_sold'].dt.to_period('M')

In [ ]:
listings['sold_price'].groupby(month_group).mean().plot(figsize = (10, 5), kind = 'line', label = 'mean')
listings['sold_price'].groupby(month_group).median().plot(figsize = (10, 5), kind = 'line', label = 'median', color = 'red')

plt.title("sold_price monthly progression (in $M)")

plt.legend()

plt.show()

In [ ]:
listings['sold_price_per_sqft'] = listings['sold_price'] / features.square_feet(listings['house_area'])

In [ ]:
listings['sold_price_per_sqft'].groupby(month_group).mean().plot(figsize = (10, 5), kind = 'line', label = 'mean')
listings['sold_price_per_sqft'].groupby(month_group).median().plot(figsize = (10, 5), kind = 'line', label = 'median', color = 'red')

plt.title("sold_price_per_sqft monthly progression (in $M)")

plt.legend()

plt.show()

In [ ]:
listings[['house_type_name', 'sold_price']].groupby([month_group, 'house_type_name'])['sold_price'].mean().unstack().plot(figsize = (10, 5))

plt.title("sold_price monthly means by house_type_name (in $M)")

plt.show()

In [ ]:
listings[['house_type_name', 'sold_price']].groupby([month_group, 'house_type_name'])['sold_price'].median().unstack().plot(figsize = (10, 5))

plt.title("sold_price monthly medians by house_type_name (in $M)")

plt.show()

In [ ]:
listings[['municipality', 'sold_price']].groupby([month_group, 'municipality'])['sold_price'].mean().unstack().plot(figsize = (10, 5))

plt.title("sold_price monthly means by municipality (in $M)")

plt.show()

In [ ]:
listings[['house_style', 'sold_price']].groupby([month_group, 'house_style'])['sold_price'].mean().unstack().plot(figsize = (10, 5))

plt.title("sold_price monthly means by house_style (in $M)")

plt.show()

In [ ]:
((listings['date_sold'] - listings['date_listed']).dt.days == listings['days_on_market']).sum()

In [ ]:
listings['remarks'].str.split().str.len()

In [ ]:
((pd.notna(listings['lot_front'])) & (pd.notna(listings['lot_depth']))).sum()

In [ ]:
len(listings[pd.notna(listings).all(axis = 1)])

In [ ]:
amenities_and_price = pd.concat([listings[['bedrooms', 'washrooms', 'parking_total', 'garage']].sum(axis = 1).astype(int),
                                 listings['sold_price']], axis = 1)

amenities_and_price.columns = ['num_amenities', 'sold_price']

amen_means = amenities_and_price.groupby('num_amenities')['sold_price'].mean()
amen_medis = amenities_and_price.groupby('num_amenities')['sold_price'].median()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (13, 5))

axes[0].scatter('num_amenities', 'sold_price', data = amenities_and_price, color = 'purple', alpha = 0.25)

axes[1].bar(x = amen_means.index, height = amen_means, color = 'yellow', alpha = 0.5, label = 'mean')
axes[1].bar(x = amen_medis.index, height = amen_medis, color = 'teal', alpha = 0.5, label = 'median')

axes[1].legend()

plt.suptitle("sold_price (points and aggregates) by total number of amenities (in $M)")
plt.show()

In [ ]:
amenities_and_price[amenities_and_price['num_amenities'] == 1].sort_values(by = ['sold_price'], ascending = False)

In [ ]:
listings['bedrooms'].astype(int).value_counts().sort_index().plot(kind = 'bar', figsize = (10, 5), color = 'lightblue', rot = 0)

plt.title("bedrooms total count distribution")

plt.show()

In [ ]:
listings['remarks_clean'] = listings['remarks'].str.lower().str.replace(r"[^\w\s]", "", regex = True)

complete_word_counts = pd.Series(Counter(' '.join(listings['remarks_clean']).split())).sort_values(ascending = False)

In [ ]:
complete_word_counts